In [ ]:
from importlib import reload
from pathlib import Path

import polars as pl

from ctm_ai_eval.utils import plots
from ctm_ai_eval.utils.io_util import load_traces_df

reload(plots)
plots.set_template()
DATASET_NAME = "general_qa_python"

# load runs & metric results
traces_file = Path(f"../tmp/traces/{DATASET_NAME}.ndjson")
metrics_file = Path(f"../tmp/metrics/{DATASET_NAME}.ndjson")

df_traces = load_traces_df(traces_file)
# indicate if rag?
df_traces = df_traces.with_columns(
    pl.col("model_spec")
    + pl.when(pl.col("rag_cfg").is_not_null()).then(pl.lit("_RAG")).otherwise(pl.lit(""))
)

df_metrics = pl.read_ndjson(metrics_file)
display(df_traces.head())
display(df_metrics.head())


In [ ]:
df_metrics.glimpse()

In [ ]:
df_wide = df_metrics.pivot(
    on="name",
    index=["run_id", "example_id"],
    values="score",
).join(df_traces.drop("server_url", "route"), on=["run_id", "example_id"])
df_wide.describe()

In [ ]:
reload(plots)
plots.group_scatter(df_wide, x="latency_ms", y="human_rating").show()
plots.group_scatter(df_wide, x="ans_length", y="latency_ms", trendline=True).show()


In [ ]:
plots.agg_mean_scatter(df_wide, y="human_rating").show()
plots.agg_mean_scatter(df_wide, y="llm_rating").show()
plots.rating_correlation(df_wide).show()